In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
import powerplantmatching as pm

from plot_helpers import (update_layout, colors)


# Conventional Plants

In [ ]:

with open("../configs/powerplantmatching_config_with_WEPP.yaml", "r") as f:
    import yaml
    config_update = yaml.safe_load(f)
    config_update["target_countries"] = ["Chile", "Morocco", "South Africa", "Egypt"]
    

def replace_natural_gas_technology(df: pd.DataFrame):
    """
    Maps and replaces gas technologies in the powerplants.csv onto model
    compliant carriers.
    """
    mapping = {
        "Steam Turbine": "CCGT",
        "Combustion Engine": "OCGT",
        "NG": "CCGT",
        "Ng": "CCGT",
        "NG/FO": "OCGT",
        "Ng/Fo": "OCGT",
        "NG/D": "OCGT",
        "LNG": "OCGT",
        "CCGT/D": "CCGT",
        "CCGT/FO": "CCGT",
        "LCCGT": "CCGT",
        "CCGT/Fo": "CCGT",
    }
    fueltype = df["Fueltype"] == "Natural Gas"
    df.loc[fueltype, "Technology"] = (
        df.loc[fueltype, "Technology"].replace(mapping).fillna("CCGT")
    )
    unique_tech_with_ng = df.loc[fueltype, "Technology"].unique()
    unknown_techs = np.setdiff1d(unique_tech_with_ng, ["CCGT", "OCGT"])
    if len(unknown_techs) > 0:
        df.loc[fueltype, "Technology"] = df.loc[fueltype, "Technology"].replace(
            {t: "CCGT" for t in unknown_techs}
        )
    df["Fueltype"] = np.where(fueltype, df["Technology"], df["Fueltype"])
    return df




In [ ]:
# pplr = pm.powerplants(from_url=False, update=True, config_update=config_update)


In [ ]:
ppl = (pplr
            .powerplant.fill_missing_decommissioning_years()
            .query('Fueltype not in ["Solar", "Wind"]')
            .powerplant.convert_country_to_alpha2()
            .pipe(replace_natural_gas_technology)
        )

# Add Age column to categorize plants by DateIn (before the lookup operation)
ppl['Age'] = ppl['DateIn'].apply(lambda x: 'Before 2015' if pd.notna(x) and x < 2015 else '2015 or later')

# Display the Age distribution
print("Age distribution:")
print(ppl['Age'].value_counts())
print(f"\nTotal plants before filtering: {len(ppl)}")

In [ ]:


powerplants_filter = "(DateOut >= 2022 or DateOut != DateOut) and Fueltype not in ['Waste', 'Hydro', 'Geothermal','Other']"

ppl = ppl.query(powerplants_filter)


In [ ]:
ppl

In [ ]:
ppl = ppl.groupby(['Country', 'Fueltype', 'Age'])["Capacity"].sum().div(1e3).reset_index()

In [ ]:


fig = px.bar(ppl, 
            x="Capacity", 
            y="Age", 
            color="Fueltype",
            facet_row="Country",
            facet_row_spacing=0.05,
            barmode="stack",
            color_discrete_map={
                "Hard Coal": colors["electricity"]["Coal"],
                "CCGT": colors["electricity"]["Gas turbines"],
                "OCGT": colors["electricity"]["CHP"],
                "Oil": colors["electricity"]["Oil"],
                "Nuclear": colors["electricity"]["Nuclear"],
            },
            # color_discrete_sequence=config_update["fhg_colors_15"],
            labels={"Capacity":"Capacity in GW", "Fueltype": "", "Age": ""},
            height=400,
            width=900,
            category_orders={"Fueltype": ["Hard Coal", "CCGT", "OCGT", "Oil", "Nuclear"], "Age": ["Before 2015", "2015 or later"]},
        )

fig.update_yaxes(matches=None)
fig.update_yaxes(showticklabels=True)
update_layout(fig)



fig.show()

In [ ]:
ppl

In [ ]:
# Calculate the share of fossil capacities going operational before 2015 per country

# Define fossil fuel types (excluding renewables and nuclear)
fossil_fuels = ["Hard Coal", "CCGT", "OCGT", "Oil"]

# Filter data for fossil fuels only
fossil_ppl = ppl[ppl['Fueltype'].isin(fossil_fuels)]

# Calculate total fossil capacity per country and age category
fossil_capacity_by_age = fossil_ppl.groupby(['Country', 'Age'])['Capacity'].sum().reset_index()

# Calculate total fossil capacity per country
total_fossil_capacity = fossil_ppl.groupby('Country')['Capacity'].sum().reset_index()
total_fossil_capacity.columns = ['Country', 'Total_Fossil_Capacity']

# Merge and calculate shares
fossil_shares = fossil_capacity_by_age.merge(total_fossil_capacity, on='Country')
fossil_shares['Share'] = (fossil_shares['Capacity'] / fossil_shares['Total_Fossil_Capacity'] * 100).round(1)

# Filter for "Before 2015" only to get the share of old fossil plants
before_2015_shares = fossil_shares[fossil_shares['Age'] == 'Before 2015'][['Country', 'Share']].copy()
before_2015_shares.columns = ['Country', 'Share_Before_2015_%']

print("Share of fossil fuel capacity that went operational before 2015 (by country):")
print("=" * 70)
for _, row in before_2015_shares.iterrows():
    print(f"{row['Country']}: {row['Share_Before_2015_%']:.1f}%")

print(f"\nSummary:")
print(f"Average share across countries: {before_2015_shares['Share_Before_2015_%'].mean():.1f}%")

# Display detailed breakdown
print("\nDetailed breakdown by country and age:")
print("=" * 50)
display(fossil_shares.pivot(index='Country', columns='Age', values='Share').fillna(0))

# Renewable plants and targets

In [ ]:
ctsn = config_update['target_countries']

In [ ]:
data = pd.read_csv("analysisdata/yearly_full_release_long_format.csv").query("Area in @ctsn and Subcategory == 'Fuel' and Variable in ['Solar','Wind'] and Category == 'Capacity' and Year >=2023")

In [ ]:
data = data.loc[:,["Area", "Variable", "Year", "Value"]]


In [ ]:
# Integrate updated South Africa solar data into the main data dataframe
print("="*80)
print("INTEGRATING updated SOLAR DATA INTO MAIN DATAFRAME")
print("="*80)

# First, let's examine the structure of the existing data dataframe
print("Original data structure:")
print(f"Columns: {list(data.columns)}")
print(f"Shape: {data.shape}")
print(f"Unique areas: {data['Area'].unique()}")
print(f"Unique variables: {data['Variable'].unique()}")
print("\nSample of original data:")
display(data.head())

# Filter out existing South Africa solar data to replace with updated values
print(f"\nFiltering out existing South Africa solar data...")
data_filtered = data[~((data['Area'] == 'South Africa') & (data['Variable'] == 'Solar'))]
print(f"Data after filtering: {data_filtered.shape}")

# Create updated South Africa solar data in the same format as the main dataframe
sa_solar_updated_formatted = pd.DataFrame({
    'Area': ['South Africa', 'South Africa'],
    'Variable': ['Solar', 'Solar'],
    'Year': [2023, 2024],
    'Value': [7.5, 8.5],  # updated values in GW
})

print("\nupdated South Africa solar data to be integrated:")
display(sa_solar_updated_formatted)

# Combine the filtered data with the updated South Africa solar data
data_updated = pd.concat([data_filtered, sa_solar_updated_formatted], ignore_index=True)

# Sort by Area, Variable, and Year for better organization
data_updated = data_updated.sort_values(['Area', 'Variable', 'Year']).reset_index(drop=True)

print(f"\nFinal updated data shape: {data_updated.shape}")
print("\nSouth Africa solar data in updated dataframe:")
sa_solar_check = data_updated[(data_updated['Area'] == 'South Africa') & (data_updated['Variable'] == 'Solar')]
display(sa_solar_check)

# Update the main data variable
datab = data_updated.copy()

print("\n" + "="*80)
print("INTEGRATION COMPLETE: Data dataframe now contains updated SA solar values")
print("="*80)

In [ ]:
data_updated["Area"] = data_updated["Area"].replace({"South Africa": "ZA", "Egypt": "EG", "Morocco": "MA", "Chile": "CL"})

In [ ]:
targetsr = pd.read_excel("analysisdata/targets_download.xlsx", sheet_name="capacity_target_wide").query("country_name in @ctsn")
targets = targetsr.query("country_name in @ctsn").copy()

In [ ]:
targets = targets.loc[:, ['country_code','target_year', 'Wind','Solar']]

In [ ]:
targets["country_code"] = ["CL", "EG", "MA", "ZA"]

In [ ]:
new_targets_ZA = pd.DataFrame({
    'country_code': ['ZA'],
    'target_year': [2030],
    'Wind': [7.2],  # IRP 2024 draft
    'Solar': [7.8+11.3]  # updated values in GW
})
targets = targets.query("country_code != 'ZA'")

targets = pd.concat([targets, new_targets_ZA], ignore_index=True)

In [ ]:
targets_long = targets.melt(
    id_vars=['country_code', 'target_year'],
    value_vars=['Wind', 'Solar'],
    var_name='Variable',
    value_name='Value'
)
targets_long

In [ ]:
targets_long.columns = ["Area", "Year", "Variable", "Value"]

In [ ]:
trend = pd.concat([data_updated, targets_long], ignore_index=True)

In [ ]:
trend["Year"] = trend["Year"].astype(str).replace({"2030": "2030 target"})

In [ ]:
print(trend)

In [ ]:
fig = px.bar(trend, 
            y="Year", 
            x="Value", 
            color="Variable",
            facet_row="Area",
            facet_row_spacing=0.05,
            barmode="stack",
            color_discrete_map={
                "Wind": colors["electricity"]["Wind onshore"],
                "Solar": colors["electricity"]["Solar PV"],
            },
            # color_discrete_sequence=config_update["fhg_colors_15"],
            labels={"Value":"Capacity in GW", "Variable": "", "Year": ""},
            height=400,
            width=900,
            category_orders={"Year": ["2023", "2024", "2030 target"], "Variable": ["Wind", "Solar"]},
        )

fig.update_yaxes(matches=None)
fig.update_yaxes(showticklabels=True)
update_layout(fig)



fig.show()

In [ ]:
print(trend)

In [ ]:
gwpt = pd.read_excel("https://docs.google.com/spreadsheets/d/1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg/export?format=xlsx&id=1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg", skiprows=4)
gwpt = gwpt[gwpt["Country/Area"].isin(ctsn)]

In [ ]:
print(gwpt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

In [ ]:
gspt = pd.read_excel("https://docs.google.com/spreadsheets/d/1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw/export?format=xlsx&id=1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw", skiprows=4)
gspt = gspt[gspt["Country/Area"].isin(ctsn)]

In [ ]:
print(gspt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

In [ ]:
# Comprehensive Analysis: Trend Data vs 2030 Targets vs GWPT/GSPT Pipeline
print("="*100)
print("COMPREHENSIVE RENEWABLE ENERGY ANALYSIS: TRENDS, TARGETS & PIPELINE")
print("="*100)

# 1. TREND ANALYSIS
print("\n1. TREND ANALYSIS (2023-2024 Performance)")
print("-" * 60)

trend_2024 = trend[trend['Year'] == '2024'].copy()
trend_2030 = trend[trend['Year'] == '2030 target'].copy()

for country in trend_2024['Area'].unique():
    country_2024 = trend_2024[trend_2024['Area'] == country]
    country_2030 = trend_2030[trend_2030['Area'] == country]
    
    print(f"\n{country}:")
    
    # Current capacities (2024)
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    
    # 2030 targets
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Calculate gaps
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    print(f"  Solar: {solar_2024:.1f} GW (2024) → {solar_2030:.1f} GW (2030) | Gap: {solar_gap:.1f} GW")
    print(f"  Wind:  {wind_2024:.1f} GW (2024) → {wind_2030:.1f} GW (2030) | Gap: {wind_gap:.1f} GW")
    print(f"  Total Gap: {solar_gap + wind_gap:.1f} GW")

# 2. PIPELINE ANALYSIS
print(f"\n\n2. PIPELINE ANALYSIS (GWPT & GSPT Prospective Capacity)")
print("-" * 60)

print("\nWind Pipeline (GWPT - Global Wind Power Tracker):")
print(gwpt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

print("\nSolar Pipeline (GSPT - Global Solar Power Tracker):")
print(gspt[["Country/Area","Prospective (Sum of Construction, Pre-construction, Announced)"]])

# 3. GAP ANALYSIS WITH PIPELINE CONTEXT
print(f"\n\n3. GAP ANALYSIS: 2030 TARGETS vs PIPELINE CAPACITY")
print("-" * 60)

# Create mapping for country names
country_mapping = {
    'ZA': 'South Africa',
    'EG': 'Egypt', 
    'MA': 'Morocco',
    'CL': 'Chile'
}

for country_code in trend_2024['Area'].unique():
    country_name = country_mapping.get(country_code, country_code)
    
    country_2024 = trend_2024[trend_2024['Area'] == country_code]
    country_2030 = trend_2030[trend_2030['Area'] == country_code]
    
    # Current and target capacities
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Required additions
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    # Pipeline capacities
    wind_pipeline = 0
    solar_pipeline = 0
    
    # Extract pipeline data
    gwpt_country = gwpt[gwpt["Country/Area"] == country_name]
    if not gwpt_country.empty:
        wind_pipeline = gwpt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].values[0]
    
    gspt_country = gspt[gspt["Country/Area"] == country_name]
    if not gspt_country.empty:
        solar_pipeline = gspt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].values[0]
    
    print(f"\n{country_name} ({country_code}):")
    print(f"  Solar Gap to 2030: {solar_gap:.1f} GW | Pipeline: {solar_pipeline:.1f} GW | Coverage: {(solar_pipeline/solar_gap*100) if solar_gap > 0 else 0:.0f}%")
    print(f"  Wind Gap to 2030:  {wind_gap:.1f} GW | Pipeline: {wind_pipeline:.1f} GW | Coverage: {(wind_pipeline/wind_gap*100) if wind_gap > 0 else 0:.0f}%")
    
    # Assessment
    if solar_pipeline >= solar_gap and wind_pipeline >= wind_gap:
        status = "✅ STRONG PIPELINE - Targets likely achievable"
    elif (solar_pipeline >= solar_gap * 0.7) and (wind_pipeline >= wind_gap * 0.7):
        status = "⚠️  MODERATE PIPELINE - Targets challenging but possible"
    else:
        status = "❌ WEAK PIPELINE - Targets at risk"
    
    print(f"  Assessment: {status}")

# 4. SUMMARY AND INSIGHTS
print(f"\n\n4. KEY INSIGHTS & SUMMARY")
print("=" * 60)

print("\n🔍 TREND OBSERVATIONS:")
print("• Solar showing consistent growth across all countries")
print("• Wind development varies significantly by country")
print("• South Africa updated solar data shows 13.3% annual growth (7.5→8.5 GW)")

print("\n🎯 TARGET FEASIBILITY:")
print("• Large capacity additions required across all countries")
print("• 6-year timeline (2024-2030) requires aggressive deployment")
print("• Pipeline data suggests varying degrees of readiness")

print("\n⚡ PIPELINE REALITY CHECK:")
print("• Construction + Pre-construction + Announced projects provide insight")
print("• High pipeline coverage suggests targets are backed by concrete projects")
print("• Low coverage indicates need for additional project development")

print("\n🚨 RISK FACTORS:")
print("• Grid integration challenges (as noted in CRSES analysis)")
print("• Transmission constraints (mentioned in Eskom GCCA 2025)")
print("• Regulatory and financing bottlenecks")
print("• Competition with aging fossil infrastructure replacement needs")

print("\n" + "="*100)